In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp, scipy.io as sio
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'rt_data').is_dir())
DATA = ROOT / 'rt_data'
CACHE = DATA / 'cache'; CACHE.mkdir(exist_ok=True)
FIGDIR = ROOT / 'explore' / 'figs'
SAVE = True
SEED = 67
if SAVE: FIGDIR.mkdir(parents=True, exist_ok=True)

INK, INK2, GRID = '#0b0b0b', '#52514e', '#e6e5e1'
mpl.rcParams.update({
    'figure.dpi': 110, 'font.size': 10, 'text.color': INK,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'grid.color': GRID, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
})

TYPE_COLORS = {
    'Undifferentiated': '#d4d3cd', 'Neutrophil': '#2a78d6', 'Monocyte': '#eb6834',
    'Baso': '#1baf7a', 'Mast': '#eda100', 'Meg': '#e87ba4', 'Erythroid': '#e34948',
    'Eos': '#6250d6', 'Other (DC/lymphoid)': '#52514e',
}
TYPES = list(TYPE_COLORS)
DAY_COLORS = {2: '#86b6ef', 4: '#2a78d6', 6: '#0d366b'}
SEQ = LinearSegmentedColormap.from_list('seq', ['#cde2fb', '#6da7ec', '#2a78d6', '#184f95', '#0d366b'])
DIV = LinearSegmentedColormap.from_list('div', ['#eb6834', '#f1e9e4', '#2a78d6'])

def save(fig, name):
    if SAVE: fig.savefig(FIGDIR / f'{name}.png', dpi=200, bbox_inches='tight')

def spring(ax, x, y, c=None, s=0.4, order=None, **kw):
    idx = np.arange(len(x)) if order is None else order
    ax.scatter(x[idx], y[idx], c=(c[idx] if c is not None and not isinstance(c, str) else c), s=s,
               linewidths=0, rasterized=True, **kw)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp_ in ax.spines.values(): sp_.set_visible(False)

def type_legend(ax, loc='center left', anchor=(1.0, 0.5), types=TYPES, ncol=1):
    h = [Line2D([], [], marker='o', ls='', color=TYPE_COLORS[t], markersize=7, label=t) for t in types]
    ax.legend(handles=h, loc=loc, bbox_to_anchor=anchor, frameon=False, ncol=ncol, fontsize=9)